In [1]:
%pwd

'/Users/harshpatel/Desktop/Projects/customer-churn-prediction/notebooks'

In [2]:
import os
os.chdir('../')

In [3]:
%pwd

'/Users/harshpatel/Desktop/Projects/customer-churn-prediction'

In [4]:
from pathlib import Path
from dataclasses import dataclass

In [5]:
@dataclass(frozen=True)
class ModelTrainingConfig:
    root_dir: Path
    training_data_path: Path
    base_model_path: Path
    tuned_model_path: Path

In [6]:
from CustomerChurnPrediction.constants import *
from CustomerChurnPrediction.utils.common import read_yaml,create_directories

In [7]:
class ConfigurationManager:
    def __init__(self,config_filepath=CONFIG_FILE_PATH):
        self.config = read_yaml(config_filepath)

        create_directories([self.config.artifacts_root])

    def get_model_training_config(self):

        config = self.config.model_training

        create_directories([config.root_dir])

        model_training_config = ModelTrainingConfig(
            root_dir=config.root_dir,
            training_data_path=config.training_data_path,
            base_model_path=config.base_model_path,
            tuned_model_path=config.tuned_model_path
        )

        return model_training_config

In [ ]:
import os
import sys
import time
import joblib
import optuna
import pandas as pd
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import recall_score
from CustomerChurnPrediction.utils.exception import CustomException
from CustomerChurnPrediction.utils.logger import logger
from CustomerChurnPrediction.entity.config_entity import ModelTrainingConfig

In [ ]:
class ModelTraining:
    """Trains a base XGBoost model and a tuned XGBoost model (via Optuna), saving both to disk."""

    THRESHOLD = 0.25

    def __init__(self, config: ModelTrainingConfig, n_trials: int = 30):
        """Sets up config and trial count."""
        self.config = config
        self.n_trials = n_trials

    def get_training_data(self) -> pd.DataFrame:
        """Loads the transformed training CSV."""
        logger.info(f"Loading training data from: {self.config.training_data_path}")
        return pd.read_csv(self.config.training_data_path)

    def split_data(self, df: pd.DataFrame, target_col: str = "Churn", test_size: float = 0.2):
        """Splits the data into train and test sets."""
        X = df.drop(columns=[target_col])
        y = df[target_col]

        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=test_size, random_state=42, stratify=y
        )
        logger.info(f"Split data: {X_train.shape[0]} train rows, {X_test.shape[0]} test rows")
        return X_train, X_test, y_train, y_test

    def get_scale_pos_weight(self, y_train) -> float:
        """Calculates the class weight used to correct for churn-class imbalance."""
        return (y_train == 0).sum() / (y_train == 1).sum()

    @staticmethod
    def save_model(model: XGBClassifier, path: str) -> None:
        """Saves a model to the given path, creating the folder if needed."""
        os.makedirs(os.path.dirname(path), exist_ok=True)
        joblib.dump(model, path)
        logger.info(f"Model saved to {path}")

    @staticmethod
    def save_model_for_github(model: XGBClassifier, filename: str = "tuned_model.pkl") -> None:
        """Saves a model into a small 'model/' folder meant to be committed to GitHub, since artifacts/ is not pushed."""
        github_dir = "model"
        os.makedirs(github_dir, exist_ok=True)
        path = os.path.join(github_dir, filename)
        joblib.dump(model, path)
        logger.info(f"Model saved for GitHub at {path}")

    def train_base_model(self, X_train, y_train) -> XGBClassifier:
        """Trains the base XGBoost model and saves it to disk."""
        scale_pos_weight = self.get_scale_pos_weight(y_train)

        params = {
            "n_estimators": 500,
            "learning_rate": 0.05,
            "max_depth": 6,
            "subsample": 0.8,
            "colsample_bytree": 0.8,
            "random_state": 42,
            "n_jobs": -1,
            "scale_pos_weight": scale_pos_weight,
            "eval_metric": "logloss",
        }

        start_train = time.time()
        model = XGBClassifier(**params)
        model.fit(X_train, y_train)
        logger.info(f"Base model trained in {time.time() - start_train:.2f} seconds")

        self.save_model(model, self.config.base_model_path)

        return model

    def tune_model(self, X_train, y_train, X_test, y_test) -> XGBClassifier:
        """Runs Optuna to find the best hyperparameters, then trains and saves the final tuned model."""
        scale_pos_weight = self.get_scale_pos_weight(y_train)

        def objective(trial: optuna.Trial) -> float:
            params = {
                "n_estimators": trial.suggest_int("n_estimators", 300, 800),
                "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2),
                "max_depth": trial.suggest_int("max_depth", 3, 10),
                "subsample": trial.suggest_float("subsample", 0.5, 1.0),
                "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
                "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
                "gamma": trial.suggest_float("gamma", 0, 5),
                "reg_alpha": trial.suggest_float("reg_alpha", 0, 5),
                "reg_lambda": trial.suggest_float("reg_lambda", 0, 5),
                "random_state": 42,
                "n_jobs": -1,
                "scale_pos_weight": scale_pos_weight,
                "eval_metric": "logloss",
            }

            model = XGBClassifier(**params)
            model.fit(X_train, y_train)
            proba = model.predict_proba(X_test)[:, 1]
            y_pred = (proba >= self.THRESHOLD).astype(int)
            return recall_score(y_test, y_pred, pos_label=1)

        study = optuna.create_study(direction="maximize")
        study.optimize(objective, n_trials=self.n_trials)

        logger.info(f"Best trial: {study.best_trial.number} | Recall: {study.best_value:.4f}")
        logger.info(f"Best params: {study.best_params}")

        best_params = {
            **study.best_params,
            "random_state": 42,
            "n_jobs": -1,
            "scale_pos_weight": scale_pos_weight,
            "eval_metric": "logloss",
        }

        start_train = time.time()
        best_model = XGBClassifier(**best_params)
        best_model.fit(X_train, y_train)
        logger.info(f"Tuned model trained in {time.time() - start_train:.2f} seconds")

        self.save_model(best_model, self.config.tuned_model_path)
        self.save_model_for_github(best_model)

        return best_model

    def run_training(self) -> None:
        """Runs the full training pipeline: load data, split, train base model, tune model."""
        df = self.get_training_data()
        X_train, X_test, y_train, y_test = self.split_data(df)

        self.train_base_model(X_train, y_train)
        self.tune_model(X_train, y_train, X_test, y_test)

In [10]:
try:
    config = ConfigurationManager()
    model_training_config = config.get_model_training_config()
    model_training = ModelTraining(model_training_config)
    model_training.run_training()
except Exception as e:
    raise CustomException(e,sys)
    

[2026-06-21 22:04:14,193] 74 CustomerChurnPrediction - INFO - yaml file: config/config.yaml loaded successfully
[2026-06-21 22:04:14,194] 91 CustomerChurnPrediction - INFO - created directory at: artifacts
[2026-06-21 22:04:14,195] 91 CustomerChurnPrediction - INFO - created directory at: artifacts/training
[2026-06-21 22:04:14,196] 13 CustomerChurnPrediction - INFO - Loading training data from: artifacts/data_transformation/transformed_data.csv
[2026-06-21 22:04:14,215] 24 CustomerChurnPrediction - INFO - Split data: 5625 train rows, 1407 test rows
[2026-06-21 22:04:14,880] 50 CustomerChurnPrediction - INFO - Base model trained in 0.66 seconds
[2026-06-21 22:04:14,886] 54 CustomerChurnPrediction - INFO - Base model saved to artifacts/training/model/base_model.pkl


[I 2026-06-21 22:04:14,888] A new study created in memory with name: no-name-bb1fe8fe-60cd-4986-b185-2b9d02cb0bfd
[I 2026-06-21 22:04:15,430] Trial 0 finished with value: 0.9278074866310161 and parameters: {'n_estimators': 673, 'learning_rate': 0.013953801196603163, 'max_depth': 5, 'subsample': 0.6396738554791674, 'colsample_bytree': 0.7153539930980257, 'min_child_weight': 4, 'gamma': 4.113696369322002, 'reg_alpha': 4.377324294671146, 'reg_lambda': 1.7430404794348138}. Best is trial 0 with value: 0.9278074866310161.
[I 2026-06-21 22:04:15,771] Trial 1 finished with value: 0.9358288770053476 and parameters: {'n_estimators': 672, 'learning_rate': 0.04044231620177849, 'max_depth': 7, 'subsample': 0.9936890978322757, 'colsample_bytree': 0.765521146057595, 'min_child_weight': 2, 'gamma': 4.356157371381248, 'reg_alpha': 0.33243010075518054, 'reg_lambda': 1.6040554151101398}. Best is trial 1 with value: 0.9358288770053476.
[I 2026-06-21 22:04:15,940] Trial 2 finished with value: 0.92780748663

[2026-06-21 22:04:25,128] 88 CustomerChurnPrediction - INFO - Best trial: 19 | Recall: 0.9439
[2026-06-21 22:04:25,128] 89 CustomerChurnPrediction - INFO - Best params: {'n_estimators': 544, 'learning_rate': 0.05119693754609395, 'max_depth': 6, 'subsample': 0.9993614013459062, 'colsample_bytree': 0.7454968868209104, 'min_child_weight': 6, 'gamma': 4.07831623847297, 'reg_alpha': 0.9617975085690853, 'reg_lambda': 4.981841592157826}
[2026-06-21 22:04:25,379] 102 CustomerChurnPrediction - INFO - Tuned model trained in 0.25 seconds
[2026-06-21 22:04:25,388] 106 CustomerChurnPrediction - INFO - Tuned model saved to artifacts/training/model/tuned_model.pkl
